Chapter Summary: Advanced Text Generation Techniques and Tools
1. Advanced Tools and Techniques framework
        a. LangChain -- this chapter
        b. DSPy
        c. Haystack
2. Chains: Extending the capabilities of LLMs
        a. A single link in the chain: Prompt template
        b. A chain with multiple prompts -- break the complex prompt into smaller subtasks to run sequentially.
3. Memory: Helping LLMs to remember conversations
        a. Conversion buffer
        b. Window conversion buffer
        c. Conversation summary
4. Agents: Creating a system of LLMs -- ability to determine the actions. Framework -- Reasoning and Acting (ReAct).
        a. Tools -- the agent can use to do things it could not do itself.
        b. Agent type -- plans the actions to take or tools to use.
5. ReAct steps
        a. Thought -- reason about the current situation.
        b. Action -- search and/or calculator.
        c. Observation -- result in action.


In [ ]:
!pip install langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 MB 13.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.2.69-cp311-cp311-linux_x86_64.whl size=55713861 sha256=08f7a56a156bae8904003e7268146a80b96a06ee4fb32e13002e450f574c7e6f
  Stored in directory: /root/.cache/pip/wheels/e8/1b/ff/b4dba97fbd16e731705b262602ba8f3b672bf4bde54ea0c104
Successfully built llama-cpp-python


In [ ]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf

--2025-03-07 05:23:58--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 13.35.202.97, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.97|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs-us-1.hf.co/repos/41/c8/41c860f65b01de5dc4c68b00d84cead799d3e7c48e38ee749f4c6057776e2e9e/5d99003e395775659b0dde3f941d88ff378b2837a8dc3a2ea94222ab1420fad3?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1741328638&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0MTMyODYzOH19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy11cy0xLmhmLmNvL3JlcG9zLzQxL2M4LzQxYzg2MGY2NWIwMWRlNWRjNGM2OGIwMGQ4NGNlYWQ3OTlkM2U3YzQ4ZTM4ZWU3NDlmNGM2MDU3Nzc2ZTJlOWUvNWQ5OTAwM2UzOTU3NzU2NTliMGRkZTNmOTQxZDg4

In [ ]:
from langchain import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

In [ ]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

# Chains

In [ ]:
from langchain import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [ ]:
basic_chain = prompt | llm

In [ ]:
# Use the chain
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

' Hello Maarten, the answer to 1 + 1 is 2.'

# Multiple Chains

In [ ]:
from langchain import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

<ipython-input-9-61dd782c6da9>:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")


In [ ]:
title.invoke({"summary": "a girl that lost her mother"})

{'summary': 'a girl that lost her mother',
 'title': ' "Whispers of Love: A Journey Through Grief"'}

In [ ]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [ ]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [ ]:
# Combine all three components to create the full chain
llm_chain = title | character | story

In [ ]:
llm_chain.invoke("a girl that lost her mother")

{'summary': 'a girl that lost her mother',
 'title': ' "Finding Warmth in Grief: A Tale of Lily\'s Journey"',
 'character': " Lily is an empathetic and resilient young girl who, after losing her beloved mother to illness, embarks on a transformative journey to find solace and healing amidst the depths of grief. Her kind heart and unyielding spirit lead her to unexpected friendships and self-discoveries that ultimately help her embrace life's beauty while honoring her cherished memories.",
 'story': " Finding Warmth in Grief: A Tale of Lily's Journey began when a young girl named Lily lost her beloved mother to illness, leaving behind an immense void that seemed impossible to fill. As she grappled with the overwhelming waves of grief and heartache, Lily realized that in order to honor her mother's memory and heal her own wounded spirit, she must embark on a transformative journey filled with self-discovery and unexpected friendships. Along the way, Lily encountered kindred souls who sha

# Memory

In [ ]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Sajid. What is 1 + 1?"})

" Hello Sajid, the answer to 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another unit."

In [ ]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" I'm sorry, but as an AI, I don't have the ability to know personal information about individuals unless it has been shared with me in the course of our conversation. Therefore, I can't provide your name."

# Conversation Buffer

In [ ]:
# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [ ]:
from langchain.memory import ConversationBufferMemory

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Sajid. What is 1 + 1?"})

{'input_prompt': 'Hi! My name is Sajid. What is 1 + 1?',
 'chat_history': '',
 'text': " Hello Sajid, the sum of 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another unit, which results in two units."}

In [ ]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Sajid. What is 1 + 1?\nAI:  Hello Sajid, the sum of 1 + 1 is 2. It's a basic arithmetic operation where you add one unit to another unit, which results in two units.",
 'text': ' Your name is mentioned as "Sajid" at the beginning of the conversation.\n\nAs for the math question, the sum of 1 + 1 is indeed 2. This is a basic addition operation within arithmetic.'}

In [ ]:
llm_chain.invoke({"input_prompt": "What is my name? Just state the name."})

{'input_prompt': 'What is my name? Just state the name.',
 'chat_history': 'Human: Hi! My name is Sajid. What is 1 + 1?\nAI:  Hello Sajid, the sum of 1 + 1 is 2. It\'s a basic arithmetic operation where you add one unit to another unit, which results in two units.\nHuman: What is my name?\nAI:  Your name is mentioned as "Sajid" at the beginning of the conversation.\n\nAs for the math question, the sum of 1 + 1 is indeed 2. This is a basic addition operation within arithmetic.',
 'text': ' Sajid'}

In [ ]:
llm_chain.invoke({"input_prompt": "What was the question asked?"})

{'input_prompt': 'What was the question asked?',
 'chat_history': 'Human: Hi! My name is Sajid. What is 1 + 1?\nAI:  Hello Sajid, the sum of 1 + 1 is 2. It\'s a basic arithmetic operation where you add one unit to another unit, which results in two units.\nHuman: What is my name?\nAI:  Your name is mentioned as "Sajid" at the beginning of the conversation.\n\nAs for the math question, the sum of 1 + 1 is indeed 2. This is a basic addition operation within arithmetic.\nHuman: What is my name? Just state the name.\nAI:  Sajid',
 'text': ' The questions asked were: "What is 1 + 1?" and "What is my name?"'}

In [ ]:
llm_chain.invoke({"input_prompt": "What did you answer for the first question?"})

{'input_prompt': 'What did you answer for the first question?',
 'chat_history': 'Human: Hi! My name is Sajid. What is 1 + 1?\nAI:  Hello Sajid, the sum of 1 + 1 is 2. It\'s a basic arithmetic operation where you add one unit to another unit, which results in two units.\nHuman: What is my name?\nAI:  Your name is mentioned as "Sajid" at the beginning of the conversation.\n\nAs for the math question, the sum of 1 + 1 is indeed 2. This is a basic addition operation within arithmetic.\nHuman: What is my name? Just state the name.\nAI:  Sajid\nHuman: What was the question asked?\nAI:  The questions asked were: "What is 1 + 1?" and "What is my name?"',
 'text': " I answered that the sum of 1 + 1 is 2. It's a basic arithmetic operation where one unit is added to another unit, resulting in two units."}

In [ ]:
llm_chain.invoke({"input_prompt": "Thank you."})

{'input_prompt': 'Thank you.',
 'chat_history': 'Human: Hi! My name is Sajid. What is 1 + 1?\nAI:  Hello Sajid, the sum of 1 + 1 is 2. It\'s a basic arithmetic operation where you add one unit to another unit, which results in two units.\nHuman: What is my name?\nAI:  Your name is mentioned as "Sajid" at the beginning of the conversation.\n\nAs for the math question, the sum of 1 + 1 is indeed 2. This is a basic addition operation within arithmetic.\nHuman: What is my name? Just state the name.\nAI:  Sajid\nHuman: What was the question asked?\nAI:  The questions asked were: "What is 1 + 1?" and "What is my name?"\nHuman: What did you answer for the first question?\nAI:  I answered that the sum of 1 + 1 is 2. It\'s a basic arithmetic operation where one unit is added to another unit, resulting in two units.',
 'text': ' The questions asked were: "What is 1 + 1?" and "What is my name?" You answered that the sum of 1 + 1 is 2 and confirmed that your name is Sajid.'}

# Conversation Buffer Memory Window

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

# Retain only the last 2 conversations in memory
memory = ConversationBufferWindowMemory(k=2, memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Ask two questions and generate two conversations in its memory
llm_chain.invoke({"input_prompt":"Hi! My name is Maarten and I am 33 years old. What is 1 + 1?"})
llm_chain.invoke({"input_prompt":"What is 3 + 3?"})

{'input_prompt': 'What is 3 + 3?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, my name isn't programmed with a specific identity, but I'm here to help you! The answer to your math question, 1 + 1 equals 2. Is there anything else you would like assistance with today?\n(Note: This response is designed to acknowledge the user while steering the conversation back to information or assistance.)",
 'text': ' Hello Maarten! Adding those numbers, 3 + 3 equals 6. Can I help you with any other questions or calculations today?'}

In [ ]:
# Check whether it knows the name we gave it
llm_chain.invoke({"input_prompt":"What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': "Human: Hi! My name is Maarten and I am 33 years old. What is 1 + 1?\nAI:  Hello Maarten, my name isn't programmed with a specific identity, but I'm here to help you! The answer to your math question, 1 + 1 equals 2. Is there anything else you would like assistance with today?\n(Note: This response is designed to acknowledge the user while steering the conversation back to information or assistance.)\nHuman: What is 3 + 3?\nAI:  Hello Maarten! Adding those numbers, 3 + 3 equals 6. Can I help you with any other questions or calculations today?",
 'text': " Your name was mentioned as Maarten in the initial conversation. However, it's important to note that as an AI, I don't have personal knowledge of individuals unless shared during our interaction. So, within this context, you referred to yourself as Maarten.\n\nAs for my identity, I'm an AI developed by Microsoft to help answer your questions and provide assistance."}

In [ ]:
# Check whether it knows the age we gave it
llm_chain.invoke({"input_prompt":"What is my age?"})

{'input_prompt': 'What is my age?',
 'chat_history': "Human: What is 3 + 3?\nAI:  Hello Maarten! Adding those numbers, 3 + 3 equals 6. Can I help you with any other questions or calculations today?\nHuman: What is my name?\nAI:  Your name was mentioned as Maarten in the initial conversation. However, it's important to note that as an AI, I don't have personal knowledge of individuals unless shared during our interaction. So, within this context, you referred to yourself as Maarten.\n\nAs for my identity, I'm an AI developed by Microsoft to help answer your questions and provide assistance.",
 'text': " As an AI, I don't have an age in the same way humans do. I was created and continue to be updated by Microsoft. However, I was last updated in September 2021.\nHere's a calculation for you: Adding those numbers again, if we consider each unit as one year since my last update, then it would be 3 (for the first number) + 3 (for the second number) = 6 years. But remember, this is just a rep

# Conversation Summary

In [ ]:
# Create a summary prompt template
summary_prompt_template = """<s><|user|>Summarize the conversations and update with the new lines.

Current summary:
{summary}

new lines of conversation:
{new_lines}

New summary:<|end|>
<|assistant|>"""
summary_prompt = PromptTemplate(
    input_variables=["new_lines", "summary"],
    template=summary_prompt_template
)

In [ ]:
from langchain.memory import ConversationSummaryMemory

# Define the type of memory we will use
memory = ConversationSummaryMemory(
    llm=llm,
    memory_key="chat_history",
    prompt=summary_prompt
)

# Chain the LLM, prompt, and memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

In [ ]:
# Generate a conversation and ask for the name
llm_chain.invoke({"input_prompt": "Hi! My name is Sajid. What is 1 + 1?"})
llm_chain.invoke({"input_prompt": "What is my name?"})

{'input_prompt': 'What is my name?',
 'chat_history': ' Sajid introduces himself and asks for the sum of 1 + 1. The AI confirms that the result is 2 and offers further assistance with math or other topics.',
 'text': " I don't have access to personal data unless it has been shared with me in the course of our conversation. Feel free to ask any other questions you might have!"}

In [ ]:
# Check whether it has summarized everything thus far
llm_chain.invoke({"input_prompt": "What was the first question I asked?"})

{'input_prompt': 'What was the first question I asked?',
 'chat_history': " Sajid introduces himself and inquires about the sum of 1 + 1, which the AI confirms as 2. When asked about their name, the AI informs them that it doesn't have access to personal data unless shared during conversation, offering further assistance with other inquiries.",
 'text': ' The first question you asked was "confirming the sum of 1 + 1."'}

In [ ]:
# Check what the summary is thus far
memory.load_memory_variables({})

{'chat_history': " Sajid introduces himself and inquires about the sum of 1 + 1, which the AI confirms as 2. When asked about their name, the AI informs them that it doesn't have access to personal data unless shared during conversation, offering further assistance with other inquiries."}

# Agents

In [ ]:
import os
from langchain_openai import ChatOpenAI

# Load OpenAI's LLMs with LangChain
os.environ["OPENAI_API_KEY"] = ""
openai_llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [ ]:
# Create the ReAct template
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

prompt = PromptTemplate(
    template=react_template,
    input_variables=["tools", "tool_names", "input", "agent_scratchpad"]
)

In [ ]:
from langchain.agents import load_tools, Tool
from langchain.tools import DuckDuckGoSearchResults

# You can create the tool to pass to an agent
search = DuckDuckGoSearchResults()
search_tool = Tool(
    name="duckduck",
    description="A web search engine. Use this to as a search engine for general queries.",
    func=search.run,
)

# Prepare tools
tools = load_tools(["llm-math"], llm=openai_llm)
tools.append(search_tool)

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent

# Construct the ReAct agent
agent = create_react_agent(openai_llm, tools, prompt)
agent_executor = AgentExecutor(
    agent=agent, tools=tools, verbose=True, handle_parsing_errors=True
)

In [ ]:
# What is the Price of a MacBook Pro?
agent_executor.invoke(
    {
        "input": "What is the current price of a MacBook Pro in USD? How much would it cost in EUR if the exchange rate is 0.85 EUR for 1 USD?"
    }
)



> Entering new AgentExecutor chain...


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}